In [ ]:
import os
import json
import ast
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
RANDOM_STATE = 42

# Split ratio
EVAL_SIZE = 0.05

# Sampling label ratio trong từng source
# 1.0 = giữ toàn bộ phishing, sample legit bằng số phishing
LEGIT_RATIO = 1.0

# Phase language ratio
# Phase 1: EN nhiều hơn VIE để học task tổng quát
PHASE1_EN_TO_VIE_RATIO = 2.0

# Phase 2: VIE-heavy + một ít EN replay để tránh quên EN
PHASE2_EN_TO_VIE_RATIO = 0.5

MAX_EVIDENCE_ITEMS = 6

In [ ]:
EMAIL_EN_PATH = "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/Dataset_Email_En_with_signals_evidence.csv"
SMS_EN_PATH = "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/Dataset_SMS_En_with_signals_evidence.csv"
EMAIL_VIE_PATH = "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/Dataset_Email_Vie_with_signals_evidence.csv"
SMS_VIE_PATH = "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/Dataset_SMS_Vie_with_signals_evidence.csv"

OUTPUT_DIR = Path("qwen_sft_output")
OUTPUT_DIR.mkdir(exist_ok=True)

PHASE1_TRAIN_PATH = OUTPUT_DIR / "phase1_train.jsonl"
PHASE1_EVAL_PATH = OUTPUT_DIR / "phase1_eval.jsonl"

PHASE2_TRAIN_PATH = OUTPUT_DIR / "phase2_train.jsonl"
PHASE2_EVAL_PATH = OUTPUT_DIR / "phase2_eval.jsonl"

DEBUG_SAMPLED_CSV_PATH = OUTPUT_DIR / "sampled_dataset_debug.csv"
DEBUG_PHASE1_CSV_PATH = OUTPUT_DIR / "phase1_dataset_debug.csv"
DEBUG_PHASE2_CSV_PATH = OUTPUT_DIR / "phase2_dataset_debug.csv"

FINAL_SIGNALS = {
    "urgency",
    "impersonation",
    "credential_request",
    "financial_lure",
    "threat",
    "call_to_action",
    "suspicious_attachment",
    "otp_or_code",
    "social_engineering",
    "domain_spoofing",
    "extortion",
}

In [ ]:
df_email_en = pd.read_csv(EMAIL_EN_PATH)
df_sms_en = pd.read_csv(SMS_EN_PATH)
df_email_vie = pd.read_csv(EMAIL_VIE_PATH)
df_sms_vie = pd.read_csv(SMS_VIE_PATH)

df_email_en["source"] = "email_en"
df_sms_en["source"] = "sms_en"
df_email_vie["source"] = "email_vie"
df_sms_vie["source"] = "sms_vie"

print("Email EN:", df_email_en.shape)
print("SMS EN:", df_sms_en.shape)
print("Email VIE:", df_email_vie.shape)
print("SMS VIE:", df_sms_vie.shape)

Email EN: (82175, 26)
SMS EN: (76033, 9)
Email VIE: (18048, 18)
SMS VIE: (10013, 10)


In [ ]:
df_sms_en[df_sms_en['label'] == 1.0].head()

,Unnamed: 0,message,label,signals,model_signals_raw,evidence,evidence_rejected,extraction_status,source
0,10,STOP,1,[],[],[],[],ok,sms_en
3,19,access>>>Credit AlertAmt: NGN 27,1,"[""financial_lure""]","[""financial_lure""]","[{""signal"": ""financial_lure"", ""exact_span"": ""C...",[],ok,sms_en
4,21,Get access to all our services using ONE code....,1,"[""call_to_action"", ""otp_or_code""]","[""call_to_action"", ""otp_or_code""]","[{""signal"": ""call_to_action"", ""exact_span"": ""D...",[],ok,sms_en
30,47,Your Viber code is: 5398.Close this message an...,1,"[""call_to_action"", ""otp_or_code""]","[""call_to_action"", ""otp_or_code""]","[{""signal"": ""otp_or_code"", ""exact_span"": ""Your...",[],ok,sms_en
31,48,Your Viber code is: 3007.Close this message an...,1,"[""call_to_action"", ""otp_or_code""]","[""call_to_action"", ""otp_or_code""]","[{""signal"": ""otp_or_code"", ""exact_span"": ""Your...",[],ok,sms_en


In [ ]:
for name, df in {
    "Email EN": df_email_en,
    "SMS EN": df_sms_en,
    "Email VIE": df_email_vie,
    "SMS VIE": df_sms_vie,
}.items():
    print(f"\n=== {name} ===")
    print("Total:", len(df))
    print(df["label"].value_counts())
    print(df["label"].value_counts(normalize=True))


=== Email EN ===
Total: 82175
label
0.0    56082
1.0    26093
Name: count, dtype: int64
label
0.0    0.68247
1.0    0.31753
Name: proportion, dtype: float64

=== SMS EN ===
Total: 76033
label
0    53396
1    22637
Name: count, dtype: int64
label
0    0.702274
1    0.297726
Name: proportion, dtype: float64

=== Email VIE ===
Total: 18048
label
0    9028
1    9020
Name: count, dtype: int64
label
0    0.500222
1    0.499778
Name: proportion, dtype: float64

=== SMS VIE ===
Total: 10013
label
1    5009
0    5004
Name: count, dtype: int64
label
1    0.50025
0    0.49975
Name: proportion, dtype: float64


Basic helper functions

In [ ]:
def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x)


def normalize_label(x):
    """
    Return:
    - 1 = phishing
    - 0 = legit
    """
    if isinstance(x, str):
        s = x.strip().lower()

        if s in {"1", "1.0", "phish", "phishing", "malicious", "spam"}:
            return 1

        if s in {"0", "0.0", "legit", "legitimate", "ham", "benign"}:
            return 0

    try:
        return 1 if int(float(x)) == 1 else 0
    except Exception:
        return 0


def parse_json_like_list(x):
    """
    Supports:
    - real list
    - JSON string
    - Python repr string
    - NaN/None
    """
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    s = str(x).strip()

    if not s or s.lower() in {"nan", "none", "null"}:
        return []

    try:
        obj = json.loads(s)
        return obj if isinstance(obj, list) else []
    except Exception:
        pass

    try:
        obj = ast.literal_eval(s)
        return obj if isinstance(obj, list) else []
    except Exception:
        return []

Text builder

In [ ]:
def first_existing(row, aliases):
    for col in aliases:
        if col in row.index:
            val = safe_str(row.get(col)).strip()
            if val:
                return val
    return ""


def build_text(row):
    """
    Ưu tiên cột text nếu có.
    Nếu không có thì build từ subject/body/message/sender/url/attachment.
    """
    direct_text = first_existing(row, ["text", "Text"])

    if direct_text:
        return direct_text.strip()

    subject = first_existing(row, ["subject", "Subject", "email_subject", "title"])
    sender = first_existing(row, ["sender", "Sender", "from", "From", "from_email"])
    body = first_existing(row, ["body", "Body", "content", "Content", "email_body"])
    message = first_existing(row, ["message", "Message", "sms", "SMS"])
    url = first_existing(row, ["url", "URL", "urls", "URLs", "link", "links", "href"])
    attachment = first_existing(row, ["attachment", "attachments", "filename", "file_name"])

    parts = []

    if subject:
        parts.append(f"Subject: {subject}")

    if sender:
        parts.append(f"Sender: {sender}")

    if body:
        parts.append(f"Body: {body}")

    if message:
        parts.append(f"Message: {message}")

    if url:
        parts.append(f"URL: {url}")

    if attachment:
        parts.append(f"Attachment: {attachment}")

    return "\n".join(parts).strip()

Evidence cleaning

In [ ]:
def simplify_evidence(evidence, max_items=MAX_EVIDENCE_ITEMS):
    """
    Chỉ giữ field cần train:
    - signal
    - exact_span
    - severity

    Không train:
    - confidence
    - char_start
    - char_end
    - verified
    - field

    Vì mấy field đó backend tự xử lý tốt hơn.
    """
    cleaned = []

    for e in evidence:
        if not isinstance(e, dict):
            continue

        signal = safe_str(e.get("signal")).strip().lower()
        exact_span = safe_str(e.get("exact_span")).strip()
        severity = safe_str(e.get("severity", "medium")).strip().lower()

        if signal not in FINAL_SIGNALS:
            continue

        if not exact_span:
            continue

        if severity not in {"low", "medium", "high"}:
            severity = "medium"

        # Nếu evidence có verified=False thì bỏ
        if e.get("verified") is False:
            continue

        cleaned.append({
            "signal": signal,
            "exact_span": exact_span,
            "severity": severity,
        })

    # Deduplicate theo signal + exact_span
    severity_rank = {
        "high": 3,
        "medium": 2,
        "low": 1,
    }

    best = {}

    for item in cleaned:
        key = (
            item["signal"],
            item["exact_span"].lower().strip(),
        )

        if key not in best:
            best[key] = item
        else:
            old = best[key]
            if severity_rank[item["severity"]] > severity_rank[old["severity"]]:
                best[key] = item

    out = list(best.values())

    # Sort high trước
    out = sorted(
        out,
        key=lambda x: severity_rank.get(x.get("severity", "medium"), 2),
        reverse=True,
    )

    return out[:max_items]


def derive_signals_from_evidence(evidence):
    """
    Signals cuối cùng lấy từ evidence.
    Như vậy signal nào không có evidence thì không train.
    """
    return sorted({
        e["signal"]
        for e in evidence
        if isinstance(e, dict) and e.get("signal") in FINAL_SIGNALS
    })

Preprocess dataset

In [ ]:
def preprocess_dataset(df, source):
    df = df.copy()
    df["source"] = source

    rows = []

    for _, row in df.iterrows():
        label = normalize_label(row.get("label"))
        text = build_text(row)

        raw_evidence = parse_json_like_list(row.get("evidence", []))
        evidence = simplify_evidence(raw_evidence)

        if label == 0:
            # Legit phải clean tuyệt đối
            signals = []
            evidence = []
        else:
            # Phishing: signals derive từ evidence
            signals = derive_signals_from_evidence(evidence)

        rows.append({
            "text": text,
            "label": label,
            "signals": signals,
            "evidence": evidence,
            "source": source,
        })

    out = pd.DataFrame(rows)

    # Bỏ text quá ngắn / empty
    out["text"] = out["text"].astype(str).str.strip()
    out = out[out["text"].str.len() >= 5].copy()

    before = len(out)

    # Vì đây là SFT label+signals+evidence,
    # phishing mà không có evidence thì bỏ khỏi train.
    out = out[~((out["label"] == 1) & (out["evidence"].apply(len) == 0))].copy()

    removed = before - len(out)

    print(f"\n=== {source} ===")
    print("Original:", len(df))
    print("After clean:", len(out))
    print("Removed phishing without evidence:", removed)
    print("\nLabel count:")
    print(out["label"].value_counts())
    print("\nLabel ratio:")
    print(out["label"].value_counts(normalize=True))

    return out.reset_index(drop=True)

Apply preprocess

In [ ]:
df_email_en_clean = preprocess_dataset(df_email_en, "email_en")
df_sms_en_clean = preprocess_dataset(df_sms_en, "sms_en")
df_email_vie_clean = preprocess_dataset(df_email_vie, "email_vie")
df_sms_vie_clean = preprocess_dataset(df_sms_vie, "sms_vie")


=== email_en ===
Original: 82175
After clean: 81402
Removed phishing without evidence: 773

Label count:
label
0    56082
1    25320
Name: count, dtype: int64

Label ratio:
label
0    0.688951
1    0.311049
Name: proportion, dtype: float64

=== sms_en ===
Original: 76033
After clean: 73897
Removed phishing without evidence: 2136

Label count:
label
0    53396
1    20501
Name: count, dtype: int64

Label ratio:
label
0    0.722573
1    0.277427
Name: proportion, dtype: float64

=== email_vie ===
Original: 18048
After clean: 18048
Removed phishing without evidence: 0

Label count:
label
0    9028
1    9020
Name: count, dtype: int64

Label ratio:
label
0    0.500222
1    0.499778
Name: proportion, dtype: float64

=== sms_vie ===
Original: 10013
After clean: 10013
Removed phishing without evidence: 0

Label count:
label
1    5009
0    5004
Name: count, dtype: int64

Label ratio:
label
1    0.50025
0    0.49975
Name: proportion, dtype: float64


In [ ]:
df_all = pd.concat([
    df_email_en_clean,
    df_sms_en_clean,
    df_email_vie_clean,
    df_sms_vie_clean,
], ignore_index=True)

df_all = df_all.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\n=== ALL CLEAN ===")
print("Total:", len(df_all))
print("\nSource x Label:")
print(pd.crosstab(df_all["source"], df_all["label"]))
print("\nLabel ratio:")
print(df_all["label"].value_counts(normalize=True))


=== ALL CLEAN ===
Total: 183360

Source x Label:
label          0      1
source                 
email_en   56082  25320
email_vie   9028   9020
sms_en     53396  20501
sms_vie     5004   5009

Label ratio:
label
0    0.673593
1    0.326407
Name: proportion, dtype: float64


Sampling: keep all phishing, downsample legit 1:1 per source

In [ ]:
def add_length_bucket(df, q=5):
    df = df.copy()
    df["text_len"] = df["text"].astype(str).str.len()

    try:
        df["len_bucket"] = pd.qcut(
            df["text_len"],
            q=q,
            labels=False,
            duplicates="drop",
        )
    except Exception:
        df["len_bucket"] = 0

    df["len_bucket"] = df["len_bucket"].fillna(0).astype(int)

    return df


def sample_legit_by_length(legit, n_target, random_state=RANDOM_STATE):
    """
    Legit thường không có signals/evidence.
    Nên stratify theo text length để giữ độ đa dạng.
    """
    if n_target >= len(legit):
        return legit.copy()

    legit = add_length_bucket(legit)

    samples = []
    total = len(legit)
    counts = legit["len_bucket"].value_counts().sort_index()

    allocated = 0
    buckets = list(counts.index)

    for i, bucket in enumerate(buckets):
        group = legit[legit["len_bucket"] == bucket]

        if i == len(buckets) - 1:
            n = n_target - allocated
        else:
            n = round(n_target * len(group) / total)
            n = min(n, len(group))
            allocated += n

        if n > 0:
            samples.append(group.sample(n=n, random_state=random_state))

    out = pd.concat(samples, ignore_index=True)

    # Fix rounding
    if len(out) > n_target:
        out = out.sample(n=n_target, random_state=random_state)

    return out.drop(columns=["text_len", "len_bucket"], errors="ignore")


def sample_keep_all_phishing_per_source(df, legit_ratio=LEGIT_RATIO):
    """
    Trong mỗi source:
    - giữ toàn bộ phishing
    - sample legit = phishing * legit_ratio
    """
    sampled_parts = []

    for source, group in df.groupby("source"):
        phish = group[group["label"] == 1].copy()
        legit = group[group["label"] == 0].copy()

        n_phish = len(phish)
        n_legit_target = min(
            len(legit),
            int(round(n_phish * legit_ratio)),
        )

        legit_sample = sample_legit_by_length(
            legit,
            n_target=n_legit_target,
            random_state=RANDOM_STATE,
        )

        out = pd.concat([phish, legit_sample], ignore_index=True)
        out = out.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

        print(f"\n=== Sampling {source} ===")
        print("Phishing kept :", len(phish))
        print("Legit sampled :", len(legit_sample), "/", len(legit))
        print("Total         :", len(out))
        print(out["label"].value_counts())
        print(out["label"].value_counts(normalize=True))

        sampled_parts.append(out)

    final = pd.concat(sampled_parts, ignore_index=True)
    final = final.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    return final

Apply source-level sampling

In [ ]:
df_sampled = sample_keep_all_phishing_per_source(
    df_all,
    legit_ratio=LEGIT_RATIO,
)

print("\n=== FINAL SAMPLED ===")
print("Total:", len(df_sampled))
print("\nSource x Label:")
print(pd.crosstab(df_sampled["source"], df_sampled["label"]))
print("\nOverall label ratio:")
print(df_sampled["label"].value_counts(normalize=True))


=== Sampling email_en ===
Phishing kept : 25320
Legit sampled : 25320 / 56082
Total         : 50640
label
1    25320
0    25320
Name: count, dtype: int64
label
1    0.5
0    0.5
Name: proportion, dtype: float64

=== Sampling email_vie ===
Phishing kept : 9020
Legit sampled : 9020 / 9028
Total         : 18040
label
1    9020
0    9020
Name: count, dtype: int64
label
1    0.5
0    0.5
Name: proportion, dtype: float64

=== Sampling sms_en ===
Phishing kept : 20501
Legit sampled : 20501 / 53396
Total         : 41002
label
1    20501
0    20501
Name: count, dtype: int64
label
1    0.5
0    0.5
Name: proportion, dtype: float64

=== Sampling sms_vie ===
Phishing kept : 5009
Legit sampled : 5004 / 5004
Total         : 10013
label
1    5009
0    5004
Name: count, dtype: int64
label
1    0.50025
0    0.49975
Name: proportion, dtype: float64

=== FINAL SAMPLED ===
Total: 119695

Source x Label:
label          0      1
source                 
email_en   25320  25320
email_vie   9020   9020
sms_en

Split EN / VIE

In [ ]:
df_en = df_sampled[
    df_sampled["source"].isin(["email_en", "sms_en"])
].copy()

df_vie = df_sampled[
    df_sampled["source"].isin(["email_vie", "sms_vie"])
].copy()

df_en = df_en.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df_vie = df_vie.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("EN:", len(df_en))
print(pd.crosstab(df_en["source"], df_en["label"]))

print("\nVIE:", len(df_vie))
print(pd.crosstab(df_vie["source"], df_vie["label"]))

EN: 91642
label         0      1
source                
email_en  25320  25320
sms_en    20501  20501

VIE: 28053
label         0     1
source               
email_vie  9020  9020
sms_vie    5004  5009


Build Phase 1 and Phase 2

In [ ]:
def build_language_mixed_phase(
    df_en,
    df_vie,
    en_to_vie_ratio,
    phase_name,
    random_state=RANDOM_STATE,
):
    """
    Build phase dataset:
    - always keep all VIE
    - sample EN according to EN:VIE ratio

    Phase 1:
      EN:VIE = 2:1

    Phase 2:
      EN:VIE = 0.5:1
    """
    n_vie = len(df_vie)

    n_en_target = min(
        len(df_en),
        int(round(n_vie * en_to_vie_ratio)),
    )

    df_en_sample = df_en.sample(
        n=n_en_target,
        random_state=random_state,
    )

    phase_df = pd.concat(
        [df_en_sample, df_vie],
        ignore_index=True,
    )

    phase_df = phase_df.sample(
        frac=1,
        random_state=random_state,
    ).reset_index(drop=True)

    print(f"\n=== {phase_name} ===")
    print("EN target:", n_en_target)
    print("VIE kept :", n_vie)
    print("Total    :", len(phase_df))
    print("\nSource x Label:")
    print(pd.crosstab(phase_df["source"], phase_df["label"]))
    print("\nLanguage count:")
    print({
        "EN": len(phase_df[phase_df["source"].isin(["email_en", "sms_en"])]),
        "VIE": len(phase_df[phase_df["source"].isin(["email_vie", "sms_vie"])]),
    })

    return phase_df

In [ ]:
phase1_df = build_language_mixed_phase(
    df_en=df_en,
    df_vie=df_vie,
    en_to_vie_ratio=PHASE1_EN_TO_VIE_RATIO,
    phase_name="PHASE 1 — General task learning",
)

phase2_df = build_language_mixed_phase(
    df_en=df_en,
    df_vie=df_vie,
    en_to_vie_ratio=PHASE2_EN_TO_VIE_RATIO,
    phase_name="PHASE 2 — Vietnamese-heavy adaptation",
)


=== PHASE 1 — General task learning ===
EN target: 56106
VIE kept : 28053
Total    : 84159

Source x Label:
label          0      1
source                 
email_en   15554  15475
email_vie   9020   9020
sms_en     12550  12527
sms_vie     5004   5009

Language count:
{'EN': 56106, 'VIE': 28053}

=== PHASE 2 — Vietnamese-heavy adaptation ===
EN target: 14026
VIE kept : 28053
Total    : 42079

Source x Label:
label         0     1
source               
email_en   3913  3781
email_vie  9020  9020
sms_en     3182  3150
sms_vie    5004  5009

Language count:
{'EN': 14026, 'VIE': 28053}


Split train/eval helper

In [ ]:
def add_split_strata(df):
    df = df.copy()
    df["num_evidence"] = df["evidence"].apply(len)

    def ev_bucket(n):
        if n == 0:
            return "0"
        if n <= 2:
            return "1_2"
        return "3plus"

    df["ev_bucket"] = df["num_evidence"].apply(ev_bucket)

    df["split_strata"] = (
        df["source"].astype(str)
        + "_"
        + df["label"].astype(str)
        + "_"
        + df["ev_bucket"].astype(str)
    )

    return df


def safe_train_eval_split(df, eval_size=EVAL_SIZE):
    df = add_split_strata(df)

    vc = df["split_strata"].value_counts()

    # Nếu strata quá nhỏ thì fallback về source + label
    if (vc < 2).any():
        df["split_strata"] = (
            df["source"].astype(str)
            + "_"
            + df["label"].astype(str)
        )

    train_df, eval_df = train_test_split(
        df,
        test_size=eval_size,
        random_state=RANDOM_STATE,
        stratify=df["split_strata"],
    )

    train_df = train_df.drop(
        columns=["num_evidence", "ev_bucket", "split_strata"],
        errors="ignore",
    )

    eval_df = eval_df.drop(
        columns=["num_evidence", "ev_bucket", "split_strata"],
        errors="ignore",
    )

    train_df = train_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    eval_df = eval_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    return train_df, eval_df

Split each phase

In [ ]:
phase1_train, phase1_eval = safe_train_eval_split(phase1_df)
phase2_train, phase2_eval = safe_train_eval_split(phase2_df)

print("Phase 1 train:", phase1_train.shape)
print("Phase 1 eval :", phase1_eval.shape)

print("\nPhase 2 train:", phase2_train.shape)
print("Phase 2 eval :", phase2_eval.shape)

Phase 1 train: (79951, 5)
Phase 1 eval : (4208, 5)

Phase 2 train: (39975, 5)
Phase 2 eval : (2104, 5)


In [ ]:
print("\n=== Phase 1 Train Source x Label ===")
print(pd.crosstab(phase1_train["source"], phase1_train["label"]))

print("\n=== Phase 1 Eval Source x Label ===")
print(pd.crosstab(phase1_eval["source"], phase1_eval["label"]))

print("\n=== Phase 2 Train Source x Label ===")
print(pd.crosstab(phase2_train["source"], phase2_train["label"]))

print("\n=== Phase 2 Eval Source x Label ===")
print(pd.crosstab(phase2_eval["source"], phase2_eval["label"]))


=== Phase 1 Train Source x Label ===
label          0      1
source                 
email_en   14776  14701
email_vie   8569   8569
sms_en     11923  11901
sms_vie     4754   4758

=== Phase 1 Eval Source x Label ===
label        0    1
source             
email_en   778  774
email_vie  451  451
sms_en     627  626
sms_vie    250  251

=== Phase 2 Train Source x Label ===
label         0     1
source               
email_en   3717  3592
email_vie  8569  8569
sms_en     3023  2992
sms_vie    4754  4759

=== Phase 2 Eval Source x Label ===
label        0    1
source             
email_en   196  189
email_vie  451  451
sms_en     159  158
sms_vie    250  250


Chat JSONL builder

In [ ]:
SYSTEM_PROMPT = """You are a cybersecurity expert for phishing detection.

Analyze an English, Vietnamese, or mixed email/SMS message and output ONLY valid JSON.

Required JSON schema:
{
  "label": "phishing" | "legit",
  "signals": ["..."],
  "evidence": [
    {
      "signal": "...",
      "exact_span": "...",
      "severity": "low" | "medium" | "high"
    }
  ]
}

Allowed signals:
urgency, impersonation, credential_request, financial_lure, threat, call_to_action, suspicious_attachment, otp_or_code, social_engineering, domain_spoofing, extortion.

Rules:
1. If the message is legitimate, output:
   {"label":"legit","signals":[],"evidence":[]}

2. If the message is phishing, output label="phishing" and include only signals supported by evidence.

3. exact_span must be copied exactly from the input message. Do not translate, paraphrase, summarize, or invent evidence.

4. evidence must be short and specific. Prefer the shortest span that clearly proves the signal.

5. Do not include signals without evidence.

6. Do not output markdown, comments, explanations, or extra keys outside the JSON.

Signal guide:
- urgency: time pressure, deadline, rushed action.
- impersonation: claiming to be a trusted brand, bank, platform, authority, support team, employer, or organization.
- credential_request: asking for password, login, OTP, code, card/bank data, wallet/private key, or identity data.
- financial_lure: reward, refund, prize, bonus, fake invoice, unexpected payment, investment profit. Not ransom/extortion.
- threat: account closure, suspension, exposure, legal action, penalty, deletion, harm, or embarrassment.
- call_to_action: instruction to click, open, download, reply, call, pay, transfer, login, verify, or send information.
- suspicious_attachment: risky or unexpected attachment/file/document.
- otp_or_code: asking to share, enter, reveal, or forward OTP/PIN/2FA/security code.
- social_engineering: fear, shame, secrecy, blackmail, fake authority, fake support, emotional pressure, greed, intimidation, or trust exploitation.
- domain_spoofing: suspicious, deceptive, lookalike, shortened, unrelated, or misleading URL/domain.
- extortion: blackmail, ransom, sextortion, or payment demand to prevent exposure, harm, or data leak.
"""


def to_assistant_json(row):
    label = "phishing" if int(row["label"]) == 1 else "legit"

    if label == "legit":
        signals = []
        evidence = []
    else:
        evidence = row["evidence"]
        signals = derive_signals_from_evidence(evidence)

    return {
        "label": label,
        "signals": signals,
        "evidence": evidence,
    }


def to_chat_record(row):
    assistant_obj = to_assistant_json(row)

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": row["text"],
            },
            {
                "role": "assistant",
                "content": json.dumps(
                    assistant_obj,
                    ensure_ascii=False,
                    separators=(",", ":"),
                ),
            },
        ]
    }


def write_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            rec = to_chat_record(row)
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

Save

In [ ]:
write_jsonl(phase1_train, PHASE1_TRAIN_PATH)
write_jsonl(phase1_eval, PHASE1_EVAL_PATH)

write_jsonl(phase2_train, PHASE2_TRAIN_PATH)
write_jsonl(phase2_eval, PHASE2_EVAL_PATH)

print("Saved:")
print(PHASE1_TRAIN_PATH)
print(PHASE1_EVAL_PATH)
print(PHASE2_TRAIN_PATH)
print(PHASE2_EVAL_PATH)

Saved:
qwen_sft_output/phase1_train.jsonl
qwen_sft_output/phase1_eval.jsonl
qwen_sft_output/phase2_train.jsonl
qwen_sft_output/phase2_eval.jsonl


In [ ]:
def save_debug_csv(df, path):
    debug_df = df.copy()

    debug_df["signals"] = debug_df["signals"].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )

    debug_df["evidence"] = debug_df["evidence"].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )

    debug_df.to_csv(path, index=False, encoding="utf-8-sig")


save_debug_csv(df_sampled, DEBUG_SAMPLED_CSV_PATH)
save_debug_csv(phase1_df, DEBUG_PHASE1_CSV_PATH)
save_debug_csv(phase2_df, DEBUG_PHASE2_CSV_PATH)

print("Saved debug CSV:")
print(DEBUG_SAMPLED_CSV_PATH)
print(DEBUG_PHASE1_CSV_PATH)
print(DEBUG_PHASE2_CSV_PATH)

In [ ]:
def validate_jsonl(path, max_check=None):
    n = 0
    bad = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            n += 1

            if max_check and n > max_check:
                break

            try:
                obj = json.loads(line)
                messages = obj["messages"]

                assert len(messages) == 3
                assert messages[0]["role"] == "system"
                assert messages[1]["role"] == "user"
                assert messages[2]["role"] == "assistant"

                assistant = json.loads(messages[2]["content"])

                assert assistant["label"] in {"phishing", "legit"}
                assert isinstance(assistant["signals"], list)
                assert isinstance(assistant["evidence"], list)

                if assistant["label"] == "legit":
                    assert assistant["signals"] == []
                    assert assistant["evidence"] == []

                if assistant["label"] == "phishing":
                    assert len(assistant["evidence"]) > 0

            except Exception as e:
                bad += 1

                if bad <= 5:
                    print("Bad line:", n)
                    print("Error:", e)
                    print(line[:500])

    print(f"{path}: checked={n}, bad={bad}")

In [ ]:
validate_jsonl(PHASE1_TRAIN_PATH, max_check=1000)
validate_jsonl(PHASE1_EVAL_PATH, max_check=1000)
validate_jsonl(PHASE2_TRAIN_PATH, max_check=1000)
validate_jsonl(PHASE2_EVAL_PATH, max_check=1000)

In [ ]:
with open(PHASE1_TRAIN_PATH, "r", encoding="utf-8") as f:
    sample = json.loads(next(f))

sample

In [ ]:
assistant_content = json.loads(sample["messages"][-1]["content"])
assistant_content

In [ ]:
def summary(df, name):
    print(f"\n=== {name} ===")
    print("Rows:", len(df))

    print("\nSource x Label:")
    print(pd.crosstab(df["source"], df["label"]))

    print("\nLabel ratio:")
    print(df["label"].value_counts(normalize=True))

    en_count = len(df[df["source"].isin(["email_en", "sms_en"])])
    vie_count = len(df[df["source"].isin(["email_vie", "sms_vie"])])

    print("\nLanguage:")
    print("EN :", en_count)
    print("VIE:", vie_count)

    if vie_count > 0:
        print("EN/VIE ratio:", round(en_count / vie_count, 3))


summary(df_sampled, "Sampled full dataset")
summary(phase1_df, "Phase 1")
summary(phase2_df, "Phase 2")
summary(phase1_train, "Phase 1 Train")
summary(phase2_train, "Phase 2 Train")

=================================Hết=====================================================================



In [ ]:
df_en_train.to_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/en_train.csv", index=False)
df_en_test.to_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/en_test.csv", index=False)

df_vn_train.to_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/vn_train.csv", index=False)
df_vn_test.to_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/vn_test.csv", index=False)

NameError: name 'df_en_train' is not defined

In [ ]:
df_en_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44997 entries, 141201 to 140687
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     44997 non-null  object
 1   label    44997 non-null  int64 
 2   signals  44997 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.4+ MB


In [ ]:
def to_chat(row):
    return {
        "messages": [
            {"role": "system", "content": """You are a cybersecurity expert specializing in phishing detection.
                                            Analyze the given message and output a JSON with:
                                            - label: "phishing" or "legit"
                                            - signals: list of detected phishing signals
                                            - reason: short explanation"""},
            {"role": "user", "content": row["text"]},
            {
                "role": "assistant",
                "content": json.dumps({
                    "label": int(row["label"]),
                    "signals": row["signals"]
                }, ensure_ascii=False)
            }
        ]
    }

In [ ]:
def save_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(json.dumps(to_chat(row), ensure_ascii=False) + "\n")

In [ ]:
# STAGE 1 (EN ONLY)
save_jsonl(df_en_train, "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/train_en.jsonl")

# STAGE 2 (VN + 20% EN)
df_en_small = df_en_train.sample(frac=0.2, random_state=SEED)

df_stage2 = pd.concat([df_vn_train, df_en_small])
df_stage2 = df_stage2.sample(frac=1, random_state=SEED).reset_index(drop=True)

save_jsonl(df_stage2, "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/train_stage2.jsonl")

# TEST EN
save_jsonl(df_en_test, "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/test_en.jsonl")

# TEST SET (REAL DISTRIBUTION)
df_test = pd.concat([df_vn_test, df_en_test])
df_test = df_test.sample(frac=1, random_state=SEED).reset_index(drop=True)

save_jsonl(df_test, "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/test.jsonl")

print("Saved JSONL files:")
print("- train_en.jsonl (Stage 1)")
print("- train_stage2.jsonl (Stage 2)")
print("- test.jsonl (Evaluation)")
print("DONE 🚀")

Saved JSONL files:
- train_en.jsonl (Stage 1)
- train_stage2.jsonl (Stage 2)
- test.jsonl (Evaluation)
DONE 🚀


In [ ]:
import json

def convert_phishing_data(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:

        for i, line in enumerate(f_in):
            data = json.loads(line)

            # 1. Tìm tin nhắn của assistant (thường là tin nhắn cuối cùng)
            for message in data['messages']:
                if message['role'] == 'assistant':
                    # 2. Parse nội dung JSON bên trong 'content'
                    try:
                        content_json = json.loads(message['content'])

                        # 3. Chuyển đổi nhãn: 0 -> legit, 1 -> phishing
                        old_label = content_json.get('label')
                        if old_label == 0:
                            content_json['label'] = "legit"
                        elif old_label == 1:
                            content_json['label'] = "phishing"

                        # 4. Cập nhật lại chuỗi string vào 'content'
                        message['content'] = json.dumps(content_json, ensure_ascii=False)
                    except json.JSONDecodeError:
                        print(f"Lỗi ở dòng {i}: Content không phải định dạng JSON chuẩn.")

            # 5. Ghi dòng đã sửa vào file mới
            f_out.write(json.dumps(data, ensure_ascii=False) + '\n')

    print(f"✅ Đã chuyển đổi xong! Check file: {output_file}")

# Thay tên file của bạn vào đây
convert_phishing_data('/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/test_en.jsonl', 'test_en.jsonl')
convert_phishing_data('/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/test.jsonl', 'test.jsonl')
convert_phishing_data('/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/train_en.jsonl', 'train_en.jsonl')
convert_phishing_data('/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/train_stage2.jsonl', 'train_stage2.jsonl')



✅ Đã chuyển đổi xong! Check file: test_en.jsonl
✅ Đã chuyển đổi xong! Check file: test.jsonl
✅ Đã chuyển đổi xong! Check file: train_en.jsonl
✅ Đã chuyển đổi xong! Check file: train_stage2.jsonl


In [2]:
import json

# Đọc file JSONL
file_path = '/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/phase2_train.jsonl'

with open(file_path, 'r', encoding='utf-8') as f:
    # Đọc tất cả các dòng
    lines = f.readlines()

# Parse từng dòng thành dict
samples = [json.loads(line) for line in lines]

# In ra 5 sample đầu tiên
for i, sample in enumerate(samples[:5]):
    print(f"Sample {i+1}:")
    print(sample)
    print("-" * 50)


Sample 1:
{'messages': [{'role': 'system', 'content': 'You are a cybersecurity expert for phishing detection.\n\nAnalyze an English, Vietnamese, or mixed email/SMS message and output ONLY valid JSON.\n\nRequired JSON schema:\n{\n  "label": "phishing" | "legit",\n  "signals": ["..."],\n  "evidence": [\n    {\n      "signal": "...",\n      "exact_span": "...",\n      "severity": "low" | "medium" | "high"\n    }\n  ]\n}\n\nAllowed signals:\nurgency, impersonation, credential_request, financial_lure, threat, call_to_action, suspicious_attachment, otp_or_code, social_engineering, domain_spoofing, extortion.\n\nRules:\n1. If the message is legitimate, output:\n   {"label":"legit","signals":[],"evidence":[]}\n\n2. If the message is phishing, output label="phishing" and include only signals supported by evidence.\n\n3. exact_span must be copied exactly from the input message. Do not translate, paraphrase, summarize, or invent evidence.\n\n4. evidence must be short and specific. Prefer the shor